# 01 - Preprocessing the LifeSnaps data

For the anomaly detection part of HealthNexus we need real wearable data to build and test on. We're using the **LifeSnaps** dataset (Yfantidou et al., 2022, *Scientific Data*), which is Fitbit data from 71 people collected over about 4 months in 2021. It's open under CC BY 4.0.

Plan for the model (so the preprocessing makes sense):
1. Start every user on a **global model** trained on everyone.
2. As a user collects more days of data, build a **personal baseline** for them and slowly switch to it.

So in this notebook we need to clean the data **and** keep track of how much history each user has on each day.

We only use the daily CSV here. The mongo dump is ~9GB of raw data and we don't need it for now.

In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
from healthnexus_ml import preprocessing as pp

pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)

## 1. Load the raw data

In [2]:
raw = pp.load_daily()
print(raw.shape)
print('users:', raw['id'].nunique())
print('dates:', raw['date'].min(), '->', raw['date'].max())
raw.head()

(7410, 62)
users: 71
dates: 2021-04-08 -> 2022-01-22


,id,date,nightly_temperature,nremhr,rmssd,spo2,full_sleep_breathing_rate,stress_score,sleep_points_percentage,exertion_points_percentage,responsiveness_points_percentage,daily_temperature_variation,badgeType,calories,filteredDemographicVO2Max,distance,activityType,bpm,lightly_active_minutes,moderately_active_minutes,...,bmi,step_goal,min_goal,max_goal,step_goal_label,ALERT,HAPPY,NEUTRAL,RESTED/RELAXED,SAD,TENSE/ANXIOUS,TIRED,ENTERTAINMENT,GYM,HOME,HOME_OFFICE,OTHER,OUTDOORS,TRANSIT,WORK/SCHOOL
0,621e2e8e67b776a24055b564,2021-05-24,34.137687,57.432,89.603,NaN,14.8,78.0,0.833333,0.675,0.866667,-1.788325,NaN,2351.59,62.79210,6517.5,['Walk'],71.701565,149.0,24.0,...,<19,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,621e2e8e67b776a24055b564,2021-05-25,33.794544,57.681,94.303,NaN,15.8,80.0,0.833333,0.725,0.866667,-2.462709,NaN,2332.08,62.67912,7178.6,['Walk'],70.579300,132.0,25.0,...,<19,14999,10000.0,15000.0,10000-14999,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,621e2e8e67b776a24055b564,2021-05-26,34.611011,57.481,119.212,NaN,14.6,84.0,0.966667,0.725,0.866667,-2.385801,NaN,2262.30,62.57307,6090.9,['Walk'],71.842573,112.0,27.0,...,<19,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,621e2e8e67b776a24055b564,2021-05-27,34.408304,57.493,111.709,NaN,14.8,82.0,0.933333,0.725,0.833333,-2.124199,['LIFETIME_DISTANCE'],2325.10,62.47493,6653.1,['Walk'],71.725477,133.0,21.0,...,<19,14999,10000.0,15000.0,10000-14999,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0
4,621e2e8e67b776a24055b564,2021-05-28,34.178922,56.750,103.034,NaN,15.2,81.0,0.866667,0.725,0.866667,-2.396873,NaN,2586.76,62.41166,9557.9,['Walk'],74.401028,136.0,42.0,...,<19,NaN,NaN,NaN,NaN,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


Rows per user. Everyone has at least 64 days, so there's enough to test the switch from the global model to a personal one.

In [3]:
raw.groupby('id').size().describe()

count     71.000000
mean     104.366197
std       45.041327
min       64.000000
25%       73.500000
50%       88.000000
75%      114.000000
max      244.000000
dtype: float64

## 2. Picking the columns

There are 63 columns. We only want the ones that say something about health and that a normal smartwatch would also give, so the model still works with our app's users later:

- **heart**: resting HR, HRV (rmssd), heart rate during deep sleep (nremhr)
- **sleep**: minutes asleep, sleep efficiency, breathing rate, skin temperature at night
- **activity**: steps, very active minutes, sedentary minutes

We're leaving out:
- `spo2` - only 17% of days have it and most users have none at all
- `stress_score` - Fitbit's own score, only 25% of days, and it's already a model output, not a raw signal
- `calories`, `bpm`, `distance` - mostly duplicate steps/HR
- age/gender/bmi - only given in brackets like `<30`, so they're not much use per user

We do keep the **mood columns** (TENSE/ANXIOUS, TIRED etc.). We won't train on them but we can use them later to check if the days we flag line up with people saying they felt bad.

In [4]:
cols_to_check = ['spo2', 'stress_score', *pp.FEATURES.keys()]
raw[cols_to_check].notna().mean().sort_values().round(2)

spo2                         0.17
stress_score                 0.25
rmssd                        0.33
nremhr                       0.33
full_sleep_breathing_rate    0.34
nightly_temperature          0.45
minutesAsleep                0.48
sleep_efficiency             0.48
resting_hr                   0.60
steps                        0.64
very_active_minutes          0.96
sedentary_minutes            0.96
dtype: float64

In [5]:
df = pp.select_columns(raw)
df.head()

,user_id,date,resting_hr,hrv_rmssd,sleep_hr,breathing_rate,skin_temp,sleep_minutes,sleep_efficiency,steps,very_active_minutes,sedentary_minutes,ALERT,HAPPY,NEUTRAL,RESTED/RELAXED,SAD,TENSE/ANXIOUS,TIRED
0,621e2e8e67b776a24055b564,2021-05-24,62.073070,89.603,57.432,14.8,34.137687,445.0,93.0,8833.0,33.0,713.0,0,0,0,0,0,0,0
1,621e2e8e67b776a24055b564,2021-05-25,62.121476,94.303,57.681,15.8,33.794544,460.0,94.0,9727.0,31.0,704.0,0,1,0,0,0,0,1
2,621e2e8e67b776a24055b564,2021-05-26,62.263999,119.212,57.481,14.6,34.611011,493.0,96.0,8253.0,31.0,710.0,0,1,0,0,0,0,0
3,621e2e8e67b776a24055b564,2021-05-27,62.368900,111.709,57.493,14.8,34.408304,540.0,93.0,9015.0,37.0,622.0,0,1,0,0,0,0,0
4,621e2e8e67b776a24055b564,2021-05-28,61.965409,103.034,56.750,15.2,34.178922,493.0,94.0,12949.0,54.0,647.0,0,1,0,0,0,0,0


## 3. Removing days the watch wasn't worn

While looking at the data we found a lot of days with **exactly 1440 sedentary minutes**, i.e. the whole day sitting still. That isn't real. It's what Fitbit records when the watch is off (charging, left at home).

In [6]:
print('days with 1440 sedentary minutes:', (df['sedentary_minutes'] >= 1440).sum(), 'of', len(df))
df[df['sedentary_minutes'] >= 1440][['steps', 'resting_hr', 'sleep_minutes']].notna().mean().round(2)

days with 1440 sedentary minutes: 2329 of 7410


steps            0.03
resting_hr       0.00
sleep_minutes    0.00
dtype: float64

Most of these days have no steps and no heart rate either. If we left them in, the model would learn that "0 steps" is normal, and then miss real drops in activity.

Rule: drop a day if it's fully sedentary **and** has no steps **and** no resting HR. If the day still has some signal we keep it, but set the 1440 to missing.

In [7]:
before = len(df)
df = pp.drop_nonwear_days(df)
print(f'dropped {before - len(df)} non-wear days, {len(df)} left')

dropped 2644 non-wear days, 4766 left


## 4. Removing impossible values

Sensors glitch sometimes. We set values outside a reasonable human range to missing (NaN) rather than deleting the whole day, since the rest of that day's readings might be fine.

The ranges are wide on purpose. We only want to catch broken readings, **not** real unusual values, because those are exactly what the anomaly model needs to find.

In [8]:
pd.DataFrame(pp.VALID_RANGES, index=['min', 'max']).T

,min,max
resting_hr,30,120
hrv_rmssd,5,200
sleep_hr,30,120
breathing_rate,6,30
skin_temp,30,38
sleep_minutes,120,900
sleep_efficiency,50,100
steps,0,60000
very_active_minutes,0,600
sedentary_minutes,0,1440


In [9]:
feat = list(pp.FEATURES.values())
before_na = df[feat].isna().sum()
df = pp.mask_invalid_values(df)
(df[feat].isna().sum() - before_na).rename('values set to NaN')

resting_hr              0
hrv_rmssd              44
sleep_hr               43
breathing_rate         87
skin_temp              25
sleep_minutes          70
sleep_efficiency       85
steps                   0
very_active_minutes     0
sedentary_minutes      10
Name: values set to NaN, dtype: int64

Also: if someone slept under 2 hours (probably a nap or the tracking cut out), the overnight values from that "night" like HRV and skin temperature aren't comparable with a full night. So we clear all the sleep columns for that day too.

After this we drop any day that has nothing left at all.

In [10]:
df = df.dropna(subset=feat, how='all')
print(df.shape)

(4766, 19)


## 5. Tracking how much history each user has

This is for the personal baseline idea. For each user and each day we add:
- `day_index` - which day of data this is for the user (1, 2, 3...)
- `days_since_start` - calendar days since their first day
- `<feature>_n_obs` - how many days with a valid value for that feature they've had **up to and including** this day

We count per feature because coverage differs a lot. Someone could have 60 days of steps but only 20 nights of HRV, so they might be ready for a personal steps baseline before a personal HRV one.

Everything only counts days up to the current day, never future ones, so when we simulate the app later there's no data leakage.

Quick check: how many users would reach 21 days of valid readings for each signal (21 is the threshold we're thinking of for switching to a personal baseline, can tune later).

In [12]:
last = df.groupby('user_id').tail(1)
pd.Series({f: (last[f'{f}_n_obs'] >= 21).sum() for f in feat}, name='users with 21+ days')

resting_hr             65
hrv_rmssd              38
sleep_hr               38
breathing_rate         38
skin_temp              53
sleep_minutes          54
sleep_efficiency       54
steps                  66
very_active_minutes    66
sedentary_minutes      66
Name: users with 21+ days, dtype: int64

## 6. Final check + save

In [13]:
print(df.shape, '| users:', df['user_id'].nunique())
df[feat].notna().mean().round(2).rename('coverage')

(4766, 31) | users: 71


resting_hr             0.93
hrv_rmssd              0.51
sleep_hr               0.51
breathing_rate         0.51
skin_temp              0.70
sleep_minutes          0.73
sleep_efficiency       0.73
steps                  0.99
very_active_minutes    1.00
sedentary_minutes      1.00
Name: coverage, dtype: float64

In [14]:
df[feat].describe().T.round(1)

,count,mean,std,min,25%,50%,75%,max
resting_hr,4422.0,66.3,7.2,44.1,61.6,66.5,71.4,86.0
hrv_rmssd,2431.0,40.5,21.0,8.3,25.7,34.2,49.0,122.1
sleep_hr,2432.0,63.5,8.5,39.7,57.6,63.3,69.3,92.5
breathing_rate,2408.0,15.3,2.7,9.8,13.4,15.2,17.2,29.8
skin_temp,3334.0,33.8,0.8,30.1,33.3,33.9,34.3,36.7
sleep_minutes,3481.0,398.6,87.4,120.0,347.0,402.0,455.0,818.0
sleep_efficiency,3466.0,94.0,5.3,50.0,93.0,95.0,97.0,100.0
steps,4711.0,8377.4,5425.1,0.0,4592.0,7498.0,11148.0,43112.0
very_active_minutes,4764.0,21.6,31.2,0.0,0.0,10.0,31.0,409.0
sedentary_minutes,4754.0,846.6,267.6,0.0,657.0,769.5,1053.0,1439.0


In [15]:
# the pipeline function runs all the steps above in one go, check it gives the same result
assert pp.preprocess().equals(df)
out = pp.save(df)
print('saved to', out)

saved to C:\Health_Nexus\ml\data\processed\daily_clean.csv


## Summary

- 7,410 raw days → **4,766 clean days**, still all 71 users
- removed non-wear days (the 1440 sedentary minutes thing)
- set impossible sensor readings to NaN instead of deleting whole days
- dropped spo2 and stress_score (too empty)
- added per-user, per-feature history counters for the global → personal baseline switch

Things we noticed for later:
- HRV / sleep HR / breathing rate are only there about half the time, so the model has to handle missing values
- One user only has 2 usable days after cleaning, so they'll just stay on the global model, which is the point of having one

**Next:** notebook 02, build the global anomaly model and the personal rolling baselines.